In [3]:
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  2.3.1
accelerate                               1.14.0
aiobotocore                              2.19.0
aiofile                                  3.11.1
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.3
aioitertools                             0.7.1
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                0.7.16
altair                                   5.5.0
anaconda-anon-usage                      0.7.1
anaconda-auth                            0.8.6
anaconda-catalogs                        0.2.0
anaconda-cli-base                        0.5.2
anaconda-client                          1.13.0
anaconda-navigator                       2.6.6
anaconda-project                         0.11.1
annotated-doc                       

In [4]:
!pip show pandas
!pip show scikit-learn
!pip show numpy

Name: pandas
Version: 2.2.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

 Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
 All rights reserved.

 Copyright (c) 2011-2023, Open source contributors.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contribut

## Model create and Save

In [ ]:
import pandas as pd
df = pd.read_csv("Cleaned_data.csv")

# train test split
from sklearn.model_selection import train_test_split
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
# task 1 transform applying
# columns transform
columns_trans = ColumnTransformer(
    [('onehot_location', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location']),
     ('onehot_area_type', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ["area_type"]),
     ('onehot_availability', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ["availability"]),
     ('scaler', StandardScaler(), ["total_sqft", "bath"]),
     ],
    remainder='passthrough')

# task 2 model apply
# model
lr = LinearRegression()
#pipeline
pipe = make_pipeline(columns_trans,lr)
print(pipe)

pipe.fit(X_train,y_train)
pipe.score(X_train,y_train)
# trainig accuracy
# Predictions
y_pred = pipe.predict(X_test)

# Performance Matrix
# Measuring Performance metrics-Lost and Cost Function (MAE,MSE,RMSE,R2 Score)
# cost functions --> calculate erros
from sklearn.metrics import mean_absolute_error, mean_squared_error,root_mean_squared_error

print("MAE:",mean_absolute_error(y_test,y_pred))
print("MSE:",mean_squared_error(y_test,y_pred))
print("RMSE:",root_mean_squared_error(y_test,y_pred))

# R2 score
from sklearn.metrics import r2_score
print(r2_score(y_test,y_pred))
# save mode
import pickle
pickle.dump(pipe,open("model_Final.pkl","wb"))

In [ ]:
d=X.sample(1)
d

## **Backend developer**

In [ ]:
# load the model
import pickle
model = pickle.load(open("model_Final.pkl","rb"))
# model.predict(d)
# def prdict():
area_type =input("enter your area type: ")
availability = input("enter your availability time :" )
location = input("Enter your loaction: ")
size_bhk = int(input ("Enter the BHK size(number of bedroom): "))
total_sqft = float(input ("Enter the total size in sqrtft: "))
bath = int(input ("Enter the bathroom(number of bathroom): "))
balcony = int(input ("Enter the balcony(number of balcony): "))

d= [[area_type,availability,location,size_bhk,total_sqft,bath,balcony]]
d

In [ ]:
X.columns

In [ ]:
d = pd.DataFrame(d,columns=X.columns)
ans = model.predict(d)
print(f"the House price based on your data is {round(ans[0],2)}₹Lakh  only")

In [ ]:
!pip install gradio

In [1]:
import pickle
import pandas as pd
import gradio as gr
# =========================
# Load Model
# =========================
with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)

# =========================
# Prediction Function
# =========================
def predict_house_price(area_type,availability,location,size_bhk,total_sqft,bath,balcony):
    try:
        # Create input data
        data = [[area_type,availability,location,int(size_bhk),float(total_sqft),int(bath),int(balcony)]]

        # IMPORTANT:
        # These column names must be exactly the same
        # as the columns used during model training.
        columns = ["area_type","availability","location","size_bhk","total_sqft","bath","balcony"]

        df = pd.DataFrame(data, columns=columns)

        # Prediction
        prediction = model.predict(df)

        price = round(prediction[0], 2)

        return f"🏠 Estimated House Price: ₹{price} Lakh"

    except Exception as e:
        return f"❌ Error: {str(e)}"
# =========================
# Gradio Interface
# =========================
with gr.Blocks(title="House Price Prediction") as app:

    gr.Markdown(
        """
        # 🏠 House Price Prediction

        Enter the property details below to predict the
        estimated house price.
        """
    )
    with gr.Row():

        with gr.Column():
            area_type = gr.Dropdown(choices=["Super built-up Area","Built-up Area","Plot Area","Carpet Area"],
                        label="Area Type",value="Super built-up Area")

            availability = gr.Textbox(label="Availability",placeholder="Example: Ready To Move")
            location = gr.Textbox(label="Location",placeholder="Example: Whitefield")
            size_bhk = gr.Number(label="BHK Size",minimum=1,value=2)

        with gr.Column():

            total_sqft = gr.Number(label="Total Size (sqft)",minimum=100,value=1000)
            bath = gr.Number(label="Number of Bathrooms",minimum=1,value=2)
            balcony = gr.Number(label="Number of Balconies",minimum=0,value=1)

    predict_button = gr.Button("Predict House Price",variant="primary")

    output = gr.Textbox(label="Prediction",interactive=False)

    predict_button.click(
        fn=predict_house_price,
        inputs=[area_type,availability,location,size_bhk,total_sqft,bath,balcony],
        outputs=output)
# =========================
# Launch App
# =========================

if __name__ == "__main__":
    app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [28]:
df

,area_type,availability,location,size_bhk,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2,1056.0,2,1,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4,2600.0,5,3,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3,1440.0,2,3,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3,1521.0,3,1,95.00
4,Super built-up Area,Ready To Move,Kothanur,2,1200.0,2,1,51.00
...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5,3453.0,4,0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4,3600.0,5,1,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2,1141.0,2,1,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4,4689.0,4,1,488.00


In [29]:
locations = df["location"].unique()
locations
# save the data for future use
pickle.dump(locations, open("locations.pkl","wb"))

# carpet area
carpet_area = df["area_type"].unique()
carpet_area
# save the data for future use
pickle.dump(carpet_area, open("area_type.pkl","wb"))

# carpet area
available = df["availability"].unique()
available
# save the data for future use
pickle.dump(available, open("avalibility.pkl","wb"))

In [30]:
locations = pickle.load(open("locations.pkl", "rb"))
area_types = pickle.load(open("area_type.pkl", "rb"))
available = pickle.load (open("avalibility.pkl","rb"))

In [ ]:
import pickle
import pandas as pd
import gradio as gr
# =========================
# Load Model
# =========================
with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)

with open("locations.pkl", "rb") as file:
    locations = pickle.load(file)
    locations = list(locations)

with open("area_type.pkl", "rb") as file:
    area_types = pickle.load(file)
    area_types= list(area_types)


with open("avalibility.pkl", "rb") as file:
    available = pickle.load(file)
    available =list(available)

# =========================
# Prediction Function
# =========================
def predict_house_price(area_type,availability,location,size_bhk,total_sqft,bath,balcony):
    try:
        # Create input data
        data = [[area_type,availability,location,int(size_bhk),float(total_sqft),int(bath),int(balcony)]]

        # IMPORTANT: xyz
        # These column names must be exactly the same
        # as the columns used during model training.
        columns = ["area_type","availability","location","size_bhk","total_sqft","bath","balcony"]

        df = pd.DataFrame(data, columns=columns)

        # Prediction
        prediction = model.predict(df)

        price = round(prediction[0], 2)

        return f"🏠 Estimated House Price: ₹{price} Lakh"

    except Exception as e:
        return f"❌ Error: {str(e)}"
# =========================
# Gradio Interface
# =========================
with gr.Blocks(title="House Price Prediction") as app:

    gr.Markdown(
        """
        # 🏠 House Price Prediction

        Enter the property details below to predict the
        estimated house price.
        """
    )
    with gr.Row():

        with gr.Column():
            area_type = gr.Dropdown(choices=area_types,
                        label="Area Type",value="Super built-up Area")

            availability = gr.Dropdown(choices=available, label="Availability")
            location = gr.Dropdown (choices= locations, label="Location")
            size_bhk = gr.Number(label="BHK Size",minimum=1,value=2)

        with gr.Column():

            total_sqft = gr.Number(label="Total Size (sqft)",minimum=100,value=1000)
            bath = gr.Number(label="Number of Bathrooms",minimum=1,value=2)
            balcony = gr.Number(label="Number of Balconies",minimum=0,value=1)

    predict_button = gr.Button("Predict House Price",variant="primary")

    output = gr.Textbox(label="Prediction",interactive=False)

    predict_button.click(
        fn=predict_house_price,
        inputs=[area_type,availability,location,size_bhk,total_sqft,bath,balcony],
        outputs=output)
# =========================
# Launch App
# =========================

if __name__ == "__main__":
    app.launch()

c:\Users\hp\anaconda3\Lib\site-packages\gradio\components\dropdown.py:235: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Super built-up Area or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Streamlit file

In [ ]:
import pickle
import pandas as pd
import streamlit as st


# =========================
# Page Configuration
# =========================

st.set_page_config(
    page_title="House Price Prediction",
    page_icon="🏠",
    layout="centered"
)


# =========================
# Load Model
# =========================

with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)


# =========================
# Load Dropdown Values
# =========================

with open("locations.pkl", "rb") as file:
    locations = pickle.load(file)
    locations = list(locations)


with open("area_type.pkl", "rb") as file:
    area_types = pickle.load(file)
    area_types = list(area_types)


with open("avalibility.pkl", "rb") as file:
    available = pickle.load(file)
    available = list(available)


# =========================
# Prediction Function
# =========================

def predict_house_price(
    area_type,
    availability,
    location,
    size_bhk,
    total_sqft,
    bath,
    balcony
):

    try:

        # Create input data
        data = [[
            area_type,
            availability,
            location,
            int(size_bhk),
            float(total_sqft),
            int(bath),
            int(balcony)
        ]]

        # Column names must match training data
        columns = [
            "area_type",
            "availability",
            "location",
            "size_bhk",
            "total_sqft",
            "bath",
            "balcony"
        ]

        # Create DataFrame
        df = pd.DataFrame(
            data,
            columns=columns
        )

        # Prediction
        prediction = model.predict(df)

        price = round(prediction[0], 2)

        return price

    except Exception as e:

        return str(e)


# =========================
# Streamlit UI
# =========================

st.title("🏠 House Price Prediction")

st.write(
    "Enter the property details below to predict "
    "the estimated house price."
)


# =========================
# Input Section
# =========================

col1, col2 = st.columns(2)


# -------- Column 1 --------

with col1:

    area_type = st.selectbox(
        "Area Type",
        options=area_types
    )

    availability = st.selectbox(
        "Availability",
        options=available
    )

    location = st.selectbox(
        "Location",
        options=locations
    )

    size_bhk = st.number_input(
        "BHK Size",
        min_value=1,
        value=2,
        step=1
    )


# -------- Column 2 --------

with col2:

    total_sqft = st.number_input(
        "Total Size (sqft)",
        min_value=100.0,
        value=1000.0,
        step=50.0
    )

    bath = st.number_input(
        "Number of Bathrooms",
        min_value=1,
        value=2,
        step=1
    )

    balcony = st.number_input(
        "Number of Balconies",
        min_value=0,
        value=1,
        step=1
    )


# =========================
# Prediction Button
# =========================

if st.button(
    "🔮 Predict House Price",
    use_container_width=True
):

    result = predict_house_price(
        area_type,
        availability,
        location,
        size_bhk,
        total_sqft,
        bath,
        balcony
    )

    # Check for error
    if isinstance(result, str):

        st.error(
            f"❌ Error: {result}"
        )

    else:

        st.success(
            f"🏠 Estimated House Price: ₹{result} Lakh"
        )